# 4-2. 토크나이징 — 한국어의 진짜 난관

> **이 노트북에서 답할 질문**
> "단어 하나"를 무엇으로 셀 것인가? 한국어에서 이게 왜 영어보다 훨씬 까다로운가?

## 1. 지금까지 우리가 쓴 것: 공백 자르기

3장과 4-1에서 `s.split()` 으로 잘랐습니다. 영어라면 그럭저럭 통합니다.
한국어에서는 어떤 일이 벌어지는지 봅시다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math, time
from collections import Counter
import torch
from common import load_corpus, load_sentences, set_seed, get_device, human

set_seed(1234)

In [ ]:
sents = load_sentences()
toks = [t for s in sents for t in s.replace(".", " .").split()]
vocab_ws = sorted(set(toks))

print(f"전체 토큰 {len(toks)}개,  서로 다른 토큰 {len(vocab_ws)}개")
print(f"토큰 하나당 평균 등장 횟수: {len(toks)/len(vocab_ws):.2f}회")

print("\n'모델' 이 들어간 토큰들:")
for w in vocab_ws:
    if "모델" in w:
        print("  ", w)

### 문제가 보이시나요

`모델은`, `모델이`, `모델을`, `모델의` … 어간은 똑같은데 **전부 다른 토큰**입니다.

모델 입장에서는 이들이 서로 아무 관계 없는 단어입니다.
원-핫이 `기계`와 `장치`를 구분 못 했던 문제(3-3)가, 이번엔 **같은 단어의 활용형 사이에서** 일어납니다.

한국어는 **교착어**라서 어간에 조사·어미가 줄줄이 붙습니다.
공백 토크나이저는 이 구조를 통째로 놓칩니다.

In [ ]:
# 어간이 같은 토큰들이 얼마나 낭비되는지 세어 보기
stems = Counter()
for w in vocab_ws:
    if len(w) >= 2:
        stems[w[:2]] += 1

dup = {k: v for k, v in stems.items() if v >= 3}
print(f"앞 2글자가 같은 토큰이 3개 이상인 어간: {len(dup)}개\n")
for k, v in sorted(dup.items(), key=lambda x: -x[1])[:8]:
    forms = [w for w in vocab_ws if w.startswith(k)][:6]
    print(f"  '{k}...' -> {v}개 : {forms}")

## 2. 반대편 극단: 글자 단위

그럼 아예 글자(음절) 하나씩 자르면 어떨까요?

In [ ]:
text = load_corpus()
chars = sorted(set(text))
print(f"글자 단위 어휘 크기: {len(chars)}  (공백 단위는 {len(vocab_ws)})")
print(f"글자 단위 시퀀스 길이: {len(text)}  (공백 단위는 {len(toks)})")
print(f"\n-> 어휘는 {len(vocab_ws)/len(chars):.1f}배 줄고, 시퀀스는 {len(text)/len(toks):.1f}배 길어집니다.")
print(f"\n어휘 일부: {''.join(chars[:40])}")

### 글자 단위의 장단점

| | 좋은 점 | 나쁜 점 |
|---|---|---|
| 어휘 크기 | 아주 작음 (수백~수천) | — |
| 처음 보는 단어 | **절대 안 생김**(모든 글자가 어휘에 있음) | — |
| 시퀀스 길이 | — | **몇 배로 길어짐** |
| 의미 단위 | — | 글자 하나에 의미가 거의 없음 |

시퀀스가 길어지는 게 왜 치명적일까요? 트랜스포머의 계산량은 길이의 **제곱**에 비례합니다(Ch5).
길이가 3배가 되면 계산량이 9배가 됩니다.

> **지금 숫자가 밋밋한 이유**: 말뭉치가 50문장뿐이라 공백 어휘(285)와 글자 어휘(253)가
> 거의 같습니다. 진짜 말뭉치라면 공백 어휘는 **수십만**으로 치솟는 반면
> 글자 어휘는 **2,000 근처에서 멈춥니다**(한글 상용 음절이 그 정도). 격차가 100배 납니다.

> **한글의 특수 사정**: 한글은 음절을 자모로 더 쪼갤 수도 있습니다(`강` → `ㄱㅏㅇ`).
> 어휘는 더 작아지지만 시퀀스는 또 3배가 됩니다. 보통은 음절 단위까지만 갑니다.

## 3. 절충안: BPE (Byte Pair Encoding)

현대 LLM이 전부 쓰는 방식입니다. 아이디어가 아주 단순합니다.

> **자주 붙어 다니는 쌍을 하나로 합치기를 반복한다.**

글자 단위에서 출발해서, 말뭉치에서 가장 빈번한 인접 쌍을 찾아 새 토큰으로 만듭니다.
이걸 원하는 어휘 크기가 될 때까지 반복합니다.

결과적으로 **자주 쓰는 표현은 통째로 한 토큰, 드문 표현은 글자로 쪼개집니다.**
`모델` 같은 어간이 하나의 토큰이 되고 `은/이/을` 이 따로 붙는 식이죠 — 우리가 원하던 바입니다.

직접 구현해 봅시다.

In [ ]:
def bpe_train(text, num_merges=200, verbose_every=50):
    """아주 단순한 BPE 학습. 글자에서 시작해 빈번한 쌍을 합쳐 나갑니다."""
    # 단어 단위로 쪼개고, 각 단어를 글자 튜플로 표현 (단어 경계를 넘어 합치지 않기 위해)
    words = Counter(text.split())
    splits = {w: tuple(w) for w in words}
    merges = []

    for i in range(num_merges):
        # 1) 인접 쌍 세기
        pairs = Counter()
        for w, freq in words.items():
            sym = splits[w]
            for j in range(len(sym) - 1):
                pairs[(sym[j], sym[j + 1])] += freq
        if not pairs:
            break

        # 2) 가장 빈번한 쌍 선택
        best, best_count = pairs.most_common(1)[0]
        if best_count < 2:          # 2번 미만이면 합칠 가치가 없음
            break
        merges.append(best)

        # 3) 모든 단어에서 그 쌍을 합치기
        merged = best[0] + best[1]
        for w, sym in splits.items():
            out, j = [], 0
            while j < len(sym):
                if j < len(sym) - 1 and (sym[j], sym[j + 1]) == best:
                    out.append(merged); j += 2
                else:
                    out.append(sym[j]); j += 1
            splits[w] = tuple(out)

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"merge {i+1:4d}: {best[0]!r} + {best[1]!r} -> {merged!r}  (빈도 {best_count})")

    return merges

merges = bpe_train(text, num_merges=200)
print(f"\n학습된 병합 규칙 {len(merges)}개")
print(f"처음 10개: {[a+b for a, b in merges[:10]]}")

In [ ]:
def bpe_encode(word, merges):
    """학습된 병합 규칙을 순서대로 적용해 한 단어를 토큰들로 자릅니다."""
    sym = list(word)
    for a, b in merges:
        out, j = [], 0
        while j < len(sym):
            if j < len(sym) - 1 and sym[j] == a and sym[j + 1] == b:
                out.append(a + b); j += 2
            else:
                out.append(sym[j]); j += 1
        sym = out
    return sym

for w in ["언어", "모델은", "모델이", "트랜스포머는", "혼란도가", "코카콜라를"]:
    print(f"{w:>8} -> {bpe_encode(w, merges)}")

print("\n--- 병합 횟수를 바꿔 가며 '모델은' 이 어떻게 잘리는지 ---")
for nm in (0, 2, 5, 8, 20, 200):
    mg = merges[:nm]
    print(f"  병합 {nm:>3}개: {bpe_encode('모델은', mg)}")

### 결과를 정직하게 읽어 봅시다

먼저 확실히 성공한 것:

- **`코카콜라를` 처럼 말뭉치에 한 번도 없던 단어도 글자로 쪼개져 처리됩니다.**
  `<unk>` 가 나올 일이 아예 없습니다. 이건 BPE의 결정적 장점입니다.
- `혼란도가` → `['혼란도', '가']` 처럼 **어간과 조사가 분리**되는 경우가 보입니다.

그런데 `모델은` 은 통째로 한 토큰이 되어 버렸습니다. 왜일까요?

아래 병합 횟수별 출력을 보세요. `모델` 은 2번째 병합에서 만들어지는데,
**9번째 병합에서 `모델` + `은` 이 다시 합쳐집니다.**
우리 말뭉치에 `모델은` 이라는 표현이 너무 자주 나오기 때문입니다.

> **이것도 일종의 과적합입니다 — 토크나이저의 과적합.**
> 말뭉치가 작으면 BPE는 "이 데이터에만 흔한 덩어리"를 통째로 외워 버립니다.
> 큰 말뭉치에서는 `모델은/모델이/모델을` 이 골고루 나오므로 `모델` 에서 병합이 멈추고,
> 조사가 독립 토큰으로 남습니다. 우리가 원하던 그림이죠.
>
> **어휘 크기와 말뭉치 크기는 함께 커져야 합니다.**

## 4. 어휘 크기의 교환비 — 실무에서 가장 중요한 감각

BPE의 병합 횟수가 곧 어휘 크기입니다. 이걸 얼마로 할지가 설계 결정입니다.

In [ ]:
n_chars = len(text.replace(" ", "").replace("\n", ""))   # 공백을 뺀 글자 수
base_chars = len(set(text.replace(" ", "")))

print(f"{'병합 횟수':>8} | {'어휘 크기':>8} | {'시퀀스 길이':>10} | {'토큰당 글자':>10}")
print("-" * 50)
for nm in (0, 50, 100, 200, 400):
    mg = bpe_train(text, num_merges=nm, verbose_every=0) if nm else []
    encoded = [t for w in text.split() for t in bpe_encode(w, mg)]
    print(f"{nm:>8} | {base_chars + len(mg):>8} | {len(encoded):>10} | {n_chars/len(encoded):>10.2f}")

print(f"\n참고 - 글자 단위: 어휘 {base_chars}, 시퀀스 {n_chars}")
print(f"참고 - 공백 단위: 어휘 {len(vocab_ws)}, 시퀀스 {len(toks)}")
print("\n400번을 요청해도 200번대에서 멈춥니다 - 합칠 만한 쌍이 더 없기 때문입니다.")
print("말뭉치가 작으면 어휘를 키우고 싶어도 키울 수가 없습니다.")

### 읽는 법

병합을 늘릴수록 **어휘는 커지고 시퀀스는 짧아집니다.** 정확히 맞바꿈입니다.

- 어휘가 크면 → 임베딩/출력층 파라미터가 커짐 (4-1에서 본 대로 V가 곱해지는 층들)
- 시퀀스가 길면 → 어텐션 계산량이 제곱으로 커짐

**양쪽 다 비용이라 중간 어딘가가 최적입니다.**

실제 모델들의 선택:

| 모델 | 어휘 크기 |
|---|---|
| GPT-2 | 50,257 |
| Llama 2 | 32,000 |
| Llama 3 | 128,256 |
| 한국어 특화 모델 | 보통 32,000 ~ 64,000 |

> **한국어가 손해를 보는 지점**: 영어 중심으로 학습된 토크나이저는 한글을
> 글자·바이트 단위로 잘게 쪼갭니다. 같은 뜻의 문장이 영어보다 토큰 수가 2~3배 나올 수 있고,
> **토큰 단위로 과금하는 API에서는 그대로 비용이 됩니다.**
> 한국어 작업에 한국어 토크나이저가 중요한 실질적 이유입니다.

## 5. 실무에서는 직접 짜지 않습니다

위 구현은 원리 이해용입니다. 실제로는 검증된 라이브러리를 씁니다.

```bash
pip install tokenizers        # Hugging Face, Rust 구현이라 빠름
pip install sentencepiece     # 구글, Llama 계열이 사용
```

```python
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tok = Tokenizer(models.BPE(unk_token=None))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
trainer = trainers.BpeTrainer(vocab_size=32000, special_tokens=["<pad>", "<s>", "</s>"])
tok.train(["corpus.txt"], trainer)
tok.save("my-tokenizer.json")
```

## 6. 진짜 한국어 말뭉치 구하기

지금까지 쓴 50문장으로는 아무것도 제대로 안 됩니다. 실제 데이터를 붙이는 방법입니다.

```python
# (a) Hugging Face 데이터셋 - 가장 간편
#     pip install datasets
from datasets import load_dataset
ds = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train")
with open("ko_wiki.txt", "w", encoding="utf-8") as f:
    for row in ds.select(range(20000)):        # 일부만
        f.write(row["text"].replace("\n", " ") + "\n")

# (b) 위키 덤프를 직접 받아 정제
#     https://dumps.wikimedia.org/kowiki/  -> wikiextractor 로 본문 추출

# (c) 그 다음은 동일하게
from common import load_sentences
sents = load_sentences("ko_wiki.txt")
```

> **M1 8GB 주의**: 위키 전체(수 GB)를 한 번에 메모리에 올리지 마세요.
> 위 예시처럼 `select(range(N))` 로 잘라 쓰거나, 파일을 줄 단위로 흘려 읽으세요.
> 실습 감을 잡는 데는 **수십 MB면 충분합니다.**

---

## 정리

| 방식 | 어휘 | 시퀀스 | 처음 보는 단어 | 한국어 적합성 |
|---|---|---|---|---|
| 공백 | 매우 큼 | 짧음 | `<unk>` 발생 | ✗ 조사마다 다른 토큰 |
| 글자 | 매우 작음 | 매우 김 | 없음 | △ 의미 단위 아님 |
| **BPE** | 조절 가능 | 중간 | 없음 | **○ 어간/조사 분리됨** |

## 연습

1. `bpe_train` 의 `num_merges` 를 1000으로 올리고 `bpe_encode("트랜스포머는")` 을 다시 보세요.
2. 큰 한국어 텍스트를 구해 BPE를 다시 학습하고, `모델/학습/훈련` 같은 어간이
   제대로 한 토큰이 되는지 확인하세요.
3. `tokenizers` 라이브러리로 같은 말뭉치를 학습시켜 우리 구현과 결과를 비교해 보세요.

## 다음

→ `0403_TrainAndGenerate.ipynb` : 제대로 된 학습 루프와, 모델이 쓴 글 읽어 보기